<a href="https://colab.research.google.com/github/mdanmek/nida-dads-notes/blob/main/dads5001-data-tools/project/eda/04_construction_review_indicators_2569.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# การตรวจรูปแบบโครงการจ้างก่อสร้าง ปีงบประมาณ 2569

ข้อมูล e-GP ปีงบประมาณ 2569 สะสมถึงวันที่ 30 กรกฎาคม 2569 ไม่ใช่ข้อมูลเต็มปี

ผลลัพธ์ใช้สำหรับจัดลำดับการตรวจเอกสาร ไม่ใช่หลักฐานว่ามีการทุจริตหรือแบ่งซื้อแบ่งจ้าง

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

!wget -q https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf
fm.fontManager.addfont('thsarabunnew-webfont.ttf')

sns.set_theme(style='whitegrid', font='TH Sarabun New')

plt.rcParams.update({
    'axes.titlesize': 18,
    'axes.titleweight': 'semibold',
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 12,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.unicode_minus': False
})

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

BLUE = '#5B7FA3'
ORANGE = '#D9822B'
GRAY = '#B8C2CC'
TEXT = '#344054'
MUTED = '#667085'
GRID = '#E4E7EC'

In [ ]:
processed_dir = Path(
    '/content/drive/MyDrive/learning/dads/dads5001/'
    'project_1_dads5001/dataset/procurement/'
    'egp-contract/processed'
)

figure_dir = processed_dir.parents[3] / 'figure'
figure_dir.mkdir(parents=True, exist_ok=True)

project_path = processed_dir / 'construction_projects_2569.csv'
contract_path = processed_dir / 'construction_contracts_2569.csv'

project_data = pd.read_csv(project_path, low_memory=False)
contract_data = pd.read_csv(contract_path, low_memory=False)

In [ ]:
project_id_column = 'รหัสโครงการ'
project_name_column = 'ชื่อโครงการจัดซื้อจัดจ้าง'
agency_column = 'ชื่อหน่วยงาน'
subagency_column = 'ชื่อหน่วยงานย่อย'
province_column = 'จังหวัด'
method_column = 'ชื่อวิธีการจัดซื้อจัดจ้าง'
budget_column = 'วงเงินงบประมาณ (บาท)'
awarded_price_column = 'ราคาที่ตกลงซื้อ / จ้าง ซึ่งรวมทุกสัญญาในโครงการ (บาท)'
transaction_date_column = 'วันที่เกิดรายการ'
contract_value_column = 'วงเงินงบประมาณในสัญญา (บาท)'
supplier_id_column = 'เลขประจำตัวนิติบุคคล 13 หลัก'
supplier_name_column = 'ชื่อผู้ชนะการเสนอราคา'

specific_method_value = 'เฉพาะเจาะจง'
legal_budget_ceiling = 500_000

project_data['in_study_scope'] = (
    project_data[method_column].eq(specific_method_value)
    & project_data[budget_column].le(legal_budget_ceiling)
)

study_project_data = project_data.loc[
    project_data['in_study_scope']
].copy()

study_project_ids = set(
    study_project_data[project_id_column]
)

study_summary = pd.Series({
    'โครงการก่อสร้างทั้งหมด': len(project_data),
    'โครงการในกลุ่มศึกษา': len(study_project_data),
    'สัดส่วนโครงการทั้งหมด (%)': (
        len(study_project_data) / len(project_data) * 100
    )
}, name='value')

display(study_summary.to_frame())

In [ ]:
project_supplier_data = (
    contract_data
    .groupby(
        [project_id_column, supplier_id_column]
    )
    .agg(
        supplier_name=(supplier_name_column, 'first'),
        supplier_awarded_value=(contract_value_column, 'sum')
    )
    .reset_index()
    .merge(
        project_data[
            [
                project_id_column,
                project_name_column,
                agency_column,
                subagency_column,
                province_column,
                method_column,
                budget_column,
                awarded_price_column,
                transaction_date_column,
                'in_study_scope'
            ]
        ],
        on=project_id_column,
        how='left'
    )
)

## 1. Pattern 1 — โครงการใกล้เพดานที่เกิดซ้ำ

ตรวจโครงการวิธีเฉพาะเจาะจง วงเงิน 490,000–500,000 บาท ที่พบในคู่หน่วยงานย่อย–ผู้รับจ้างเดียวกันอย่างน้อย 3 โครงการ

วันที่เกิดรายการไม่นำมาเป็นเงื่อนไข เพราะการคลาดกันเพียง 1 วันทำให้หลุดจาก Pattern ได้ วันที่จะแสดงไว้ในตารางรายโครงการเพื่อใช้เปิดเอกสารประกอบเท่านั้น


In [ ]:
valid_cluster_rows = (
    project_supplier_data[subagency_column].notna()
    & project_supplier_data[supplier_id_column].notna()
)

near_ceiling_data = project_supplier_data.loc[
    project_supplier_data['in_study_scope']
    & valid_cluster_rows
    & project_supplier_data[budget_column].between(
        490_000,
        legal_budget_ceiling,
        inclusive='both'
    )
].copy()

cluster_columns = [
    subagency_column,
    supplier_id_column
]

near_ceiling_clusters = (
    near_ceiling_data
    .groupby(cluster_columns, dropna=False)
    .agg(
        supplier_name=('supplier_name', 'first'),
        project_count=(project_id_column, 'nunique'),
        total_budget=(budget_column, 'sum')
    )
    .reset_index()
)

repeated_clusters = (
    near_ceiling_clusters.loc[
        near_ceiling_clusters['project_count'].ge(3)
    ]
    .sort_values(
        ['project_count', 'total_budget'],
        ascending=False
    )
    .reset_index(drop=True)
)

pattern1_project_ids = set(
    near_ceiling_data
    .merge(
        repeated_clusters[cluster_columns],
        on=cluster_columns,
        how='inner'
    )[project_id_column]
)

project_data['flag_pattern_1'] = (
    project_data[project_id_column].isin(pattern1_project_ids)
)

pattern1_summary = pd.Series({
    'คู่หน่วยงาน–ผู้รับจ้างที่ผ่านเกณฑ์': len(repeated_clusters),
    'โครงการที่เกี่ยวข้อง': project_data['flag_pattern_1'].sum(),
    'สัดส่วนกลุ่มศึกษา (%)': (
        project_data['flag_pattern_1'].sum()
        / len(study_project_data)
        * 100
    )
}, name='ค่า')

pattern1_table = (
    repeated_clusters[
        [
            subagency_column,
            'supplier_name',
            'project_count',
            'total_budget'
        ]
    ]
    .rename(columns={
        subagency_column: 'หน่วยงานย่อย',
        'supplier_name': 'ผู้รับจ้าง',
        'project_count': 'จำนวนโครงการ',
        'total_budget': 'วงเงินรวม (บาท)'
    })
)

display(pattern1_summary.to_frame())
display(pattern1_table.head(15))


### อ่านตาราง Pattern 1

แต่ละแถวคือคู่หน่วยงานย่อย–ผู้รับจ้างหนึ่งคู่ จำนวนโครงการนับเฉพาะโครงการวิธีเฉพาะเจาะจงที่มีวงเงิน 490,000–500,000 บาท โดยรวมทุกวันที่เกิดรายการ

ตารางเรียงจากคู่ที่มีจำนวนโครงการมากที่สุด เพื่อให้เห็นกลุ่มที่ควรเปิดดูรายละเอียดก่อน


## 2. Pattern 2 — การพึ่งพาผู้รับจ้างภายในหน่วยงาน: ชื่อหน่วยงาน

ตรวจคู่ชื่อหน่วยงาน–ผู้รับจ้างที่มีอย่างน้อย 10 โครงการ และผู้รับจ้างครองทั้งจำนวนโครงการและวงเงินอย่างน้อย 75% ของชื่อหน่วยงาน


In [ ]:
study_supplier_data = project_supplier_data.loc[
    project_supplier_data['in_study_scope']
    & project_supplier_data[supplier_id_column].notna()
].copy()

def summarize_dependence(group_column):
    valid_data = study_supplier_data.loc[
        study_supplier_data[group_column].notna()
        & ~study_supplier_data[group_column]
        .astype('string')
        .str.strip()
        .isin(['', '-', 'ไม่ระบุ'])
    ]

    relationships = (
        valid_data
        .groupby([group_column, supplier_id_column], dropna=False)
        .agg(
            supplier_name=('supplier_name', 'first'),
            supplier_project_count=(project_id_column, 'nunique'),
            supplier_awarded_value=('supplier_awarded_value', 'sum')
        )
        .reset_index()
    )

    totals = (
        study_project_data.loc[
            study_project_data[group_column].notna()
            & ~study_project_data[group_column]
            .astype('string')
            .str.strip()
            .isin(['', '-', 'ไม่ระบุ'])
        ]
        .groupby(group_column, dropna=False)
        .agg(
            organization_project_count=(project_id_column, 'nunique'),
            organization_awarded_value=(awarded_price_column, 'sum')
        )
        .reset_index()
    )

    relationships = relationships.merge(
        totals,
        on=group_column,
        how='left'
    )

    relationships['project_share_pct'] = (
        relationships['supplier_project_count']
        / relationships['organization_project_count']
        * 100
    )

    relationships['value_share_pct'] = (
        relationships['supplier_awarded_value']
        / relationships['organization_awarded_value']
        * 100
    )

    return relationships.rename(
        columns={group_column: 'organization_name'}
    )

def dependence_table(data, organization_label):
    return (
        data[
            [
                'organization_name',
                'supplier_name',
                'supplier_project_count',
                'organization_project_count',
                'project_share_pct',
                'supplier_awarded_value',
                'organization_awarded_value',
                'value_share_pct'
            ]
        ]
        .round({
            'project_share_pct': 2,
            'value_share_pct': 2
        })
        .rename(columns={
            'organization_name': organization_label,
            'supplier_name': 'ผู้รับจ้าง',
            'supplier_project_count': 'โครงการของผู้รับจ้าง',
            'organization_project_count': 'โครงการทั้งหมด',
            'project_share_pct': 'สัดส่วนจำนวนโครงการ (%)',
            'supplier_awarded_value': 'วงเงินของผู้รับจ้าง (บาท)',
            'organization_awarded_value': 'วงเงินรวม (บาท)',
            'value_share_pct': 'สัดส่วนวงเงิน (%)'
        })
    )


In [ ]:
agency_relationships = summarize_dependence(agency_column)

pattern2_relationships = (
    agency_relationships.loc[
        agency_relationships['supplier_project_count'].ge(10)
        & agency_relationships['project_share_pct'].ge(75)
        & agency_relationships['value_share_pct'].ge(75)
    ]
    .sort_values(
        ['project_share_pct', 'supplier_project_count'],
        ascending=False
    )
    .reset_index(drop=True)
)

pattern2_context = (
    study_supplier_data
    .merge(
        pattern2_relationships[
            [
                'organization_name',
                supplier_id_column,
                'supplier_project_count',
                'organization_project_count',
                'project_share_pct',
                'value_share_pct'
            ]
        ],
        left_on=[agency_column, supplier_id_column],
        right_on=['organization_name', supplier_id_column],
        how='inner'
    )
    .assign(pattern_name='Pattern 2')
)

pattern2_project_ids = set(
    pattern2_context[project_id_column]
)

pattern2_summary = pd.Series({
    'คู่ชื่อหน่วยงาน–ผู้รับจ้างที่ผ่านเกณฑ์': len(pattern2_relationships),
    'โครงการที่เกี่ยวข้อง': len(pattern2_project_ids)
}, name='ค่า')

display(pattern2_summary.to_frame())
display(
    dependence_table(
        pattern2_relationships,
        'ชื่อหน่วยงาน'
    ).head(20)
)


### อ่านตาราง Pattern 2

แต่ละแถวคือคู่ชื่อหน่วยงาน–ผู้รับจ้าง ตารางแสดงทั้งจำนวนโครงการและวงเงินของผู้รับจ้างเทียบกับยอดรวมของชื่อหน่วยงาน และเรียงตามสัดส่วนจำนวนโครงการจากมากไปน้อย


## 3. Pattern 3 — การพึ่งพาผู้รับจ้างภายในหน่วยงาน: ชื่อหน่วยงานย่อย

ตรวจคู่ชื่อหน่วยงานย่อย–ผู้รับจ้างที่มีอย่างน้อย 10 โครงการ และผู้รับจ้างครองทั้งจำนวนโครงการและวงเงินอย่างน้อย 75% ของชื่อหน่วยงานย่อย


In [ ]:
subagency_relationships = summarize_dependence(
    subagency_column
)

pattern3_relationships = (
    subagency_relationships.loc[
        subagency_relationships['supplier_project_count'].ge(10)
        & subagency_relationships['project_share_pct'].ge(75)
        & subagency_relationships['value_share_pct'].ge(75)
    ]
    .sort_values(
        ['project_share_pct', 'supplier_project_count'],
        ascending=False
    )
    .reset_index(drop=True)
)

pattern3_context = (
    study_supplier_data
    .merge(
        pattern3_relationships[
            [
                'organization_name',
                supplier_id_column,
                'supplier_project_count',
                'organization_project_count',
                'project_share_pct',
                'value_share_pct'
            ]
        ],
        left_on=[subagency_column, supplier_id_column],
        right_on=['organization_name', supplier_id_column],
        how='inner'
    )
    .assign(pattern_name='Pattern 3')
)

pattern3_project_ids = set(
    pattern3_context[project_id_column]
)


In [ ]:
pattern3_summary = pd.Series({
    'คู่ชื่อหน่วยงานย่อย–ผู้รับจ้างที่ผ่านเกณฑ์': (
        len(pattern3_relationships)
    ),
    'โครงการที่เกี่ยวข้อง': len(pattern3_project_ids)
}, name='ค่า')

display(pattern3_summary.to_frame())
display(
    dependence_table(
        pattern3_relationships,
        'ชื่อหน่วยงานย่อย'
    ).head(20)
)


### อ่านตาราง Pattern 3

แต่ละแถวคือคู่ชื่อหน่วยงานย่อย–ผู้รับจ้าง ตารางใช้ตัวหารจากโครงการและวงเงินรวมของชื่อหน่วยงานย่อยนั้น และเรียงตามสัดส่วนจำนวนโครงการจากมากไปน้อย


## 4. Pattern 4 — การพึ่งพาผู้รับจ้างภายในหน่วยงาน: จังหวัด

ตรวจคู่จังหวัด–ผู้รับจ้าง โดยพิจารณาจังหวัดที่มีอย่างน้อย 20 โครงการ ผู้รับจ้างได้รับอย่างน้อย 5 โครงการ และครองทั้งจำนวนโครงการและวงเงินอย่างน้อย 50% ของจังหวัด


In [ ]:
province_relationships = summarize_dependence(
    province_column
)

eligible_province_relationships = (
    province_relationships.loc[
        province_relationships['organization_project_count'].ge(20)
        & province_relationships['supplier_project_count'].ge(5)
    ]
)

pattern4_relationships = (
    eligible_province_relationships.loc[
        eligible_province_relationships['project_share_pct'].ge(50)
        & eligible_province_relationships['value_share_pct'].ge(50)
    ]
    .sort_values(
        ['project_share_pct', 'supplier_project_count'],
        ascending=False
    )
    .reset_index(drop=True)
)

pattern4_project_ids = set(
    study_supplier_data
    .merge(
        pattern4_relationships[
            ['organization_name', supplier_id_column]
        ],
        left_on=[province_column, supplier_id_column],
        right_on=['organization_name', supplier_id_column],
        how='inner'
    )[project_id_column]
)


In [ ]:
pattern4_summary = pd.Series({
    'คู่จังหวัด–ผู้รับจ้างที่ผ่านเกณฑ์': len(pattern4_relationships),
    'โครงการที่เกี่ยวข้อง': len(pattern4_project_ids),
    'สัดส่วนจำนวนโครงการสูงสุด (%)': (
        eligible_province_relationships['project_share_pct'].max()
    ),
    'สัดส่วนวงเงินสูงสุด (%)': (
        eligible_province_relationships['value_share_pct'].max()
    )
}, name='ค่า')

display(pattern4_summary.to_frame())
display(
    dependence_table(
        eligible_province_relationships.sort_values(
            'project_share_pct',
            ascending=False
        ),
        'จังหวัด'
    ).head(20)
)


### อ่านตาราง Pattern 4

แต่ละแถวคือคู่จังหวัด–ผู้รับจ้าง ตารางแสดงคู่ที่มีขนาดข้อมูลเพียงพอก่อน แล้วเรียงตามสัดส่วนจำนวนโครงการจากมากไปน้อย ผลที่ผ่านเกณฑ์จริงสรุปอยู่ในตารางด้านบน


## 5. โครงการตรวจสอบลำดับแรก

ซ้อนทับ Pattern 1 กับการพึ่งพาผู้รับจ้างใน Pattern 2 หรือ Pattern 3 ส่วน Pattern 4 แสดงเป็นบริบทระดับจังหวัด


In [ ]:
pattern1_context = (
    near_ceiling_data
    .merge(
        repeated_clusters[
            cluster_columns + ['project_count']
        ].rename(columns={
            'project_count': 'cluster_project_count'
        }),
        on=cluster_columns,
        how='inner'
    )
)

dependence_context = pd.concat(
    [pattern2_context, pattern3_context],
    ignore_index=True
)

priority_context = (
    pattern1_context
    .merge(
        dependence_context[
            [
                project_id_column,
                supplier_id_column,
                'pattern_name',
                'organization_name',
                'supplier_project_count',
                'organization_project_count',
                'project_share_pct',
                'value_share_pct'
            ]
        ],
        on=[project_id_column, supplier_id_column],
        how='inner'
    )
)

priority_project_ids = set(
    priority_context[project_id_column]
)

study_review_data = project_data.loc[
    project_data['in_study_scope']
].copy()

study_review_data['flag_pattern_1'] = (
    study_review_data[project_id_column].isin(pattern1_project_ids)
)
study_review_data['flag_pattern_2'] = (
    study_review_data[project_id_column].isin(pattern2_project_ids)
)
study_review_data['flag_pattern_3'] = (
    study_review_data[project_id_column].isin(pattern3_project_ids)
)
study_review_data['flag_pattern_4'] = (
    study_review_data[project_id_column].isin(pattern4_project_ids)
)

study_review_data['priority_review'] = (
    study_review_data[project_id_column]
    .isin(priority_project_ids)
)

result_summary = pd.DataFrame({
    'เงื่อนไข': [
        'Pattern 1: เกิดซ้ำใกล้เพดาน',
        'Pattern 2: พึ่งพาในชื่อหน่วยงาน',
        'Pattern 3: พึ่งพาในชื่อหน่วยงานย่อย',
        'Pattern 4: พึ่งพาในจังหวัด',
        'โครงการตรวจสอบลำดับแรก'
    ],
    'จำนวนโครงการ': [
        len(pattern1_project_ids),
        len(pattern2_project_ids),
        len(pattern3_project_ids),
        len(pattern4_project_ids),
        len(priority_project_ids)
    ]
})

display(result_summary)


In [ ]:
plot_data = result_summary.set_index(
    'เงื่อนไข'
)['จำนวนโครงการ']

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    plot_data.index,
    plot_data.values,
    color=[BLUE, BLUE, BLUE, BLUE, ORANGE],
    height=0.58
)

ax.bar_label(
    bars,
    labels=[f'{value:,.0f}' for value in plot_data.values],
    padding=5,
    fontsize=11,
    color=TEXT
)

ax.invert_yaxis()
ax.set_xlim(0, max(plot_data.max() * 1.18, 1))
ax.set_title(
    'จำนวนโครงการตามเงื่อนไขการตรวจสอบ',
    loc='left',
    pad=12,
    color=TEXT
)
ax.set_xlabel('จำนวนโครงการ')
ax.set_ylabel('')

ax.grid(axis='x', color=GRID, linewidth=0.8)
ax.grid(axis='y', visible=False)
ax.set_axisbelow(True)
sns.despine(left=True, bottom=True)

fig.tight_layout()

png_path = figure_dir / 'fig04_01_review_conditions.png'
svg_path = figure_dir / 'fig04_01_review_conditions.svg'

fig.savefig(png_path, dpi=180, bbox_inches='tight', facecolor='white')
fig.savefig(svg_path, bbox_inches='tight', facecolor='white')

print(f'Saved: {png_path}')
print(f'Saved: {svg_path}')


In [ ]:
priority_projects = (
    priority_context[
        [
            project_id_column,
            project_name_column,
            agency_column,
            subagency_column,
            province_column,
            'pattern_name',
            'organization_name',
            'supplier_name',
            budget_column,
            transaction_date_column,
            'cluster_project_count',
            'project_share_pct',
            'value_share_pct'
        ]
    ]
    .sort_values(
        [
            'project_share_pct',
            'cluster_project_count'
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

priority_groups = (
    priority_context
    .groupby(
        [
            'pattern_name',
            'organization_name',
            supplier_id_column,
            'supplier_name'
        ],
        dropna=False
    )
    .agg(
        priority_project_count=(project_id_column, 'nunique'),
        supplier_project_count=('supplier_project_count', 'max'),
        organization_project_count=('organization_project_count', 'max'),
        project_share_pct=('project_share_pct', 'max'),
        value_share_pct=('value_share_pct', 'max')
    )
    .reset_index()
    .sort_values(
        ['project_share_pct', 'priority_project_count'],
        ascending=False
    )
    .reset_index(drop=True)
)

priority_group_table = (
    priority_groups
    .round({
        'project_share_pct': 2,
        'value_share_pct': 2
    })
    .rename(columns={
        'pattern_name': 'Pattern',
        'organization_name': 'หน่วยงาน',
        'supplier_name': 'ผู้รับจ้าง',
        'priority_project_count': 'โครงการตรวจสอบลำดับแรก',
        'supplier_project_count': 'โครงการของผู้รับจ้าง',
        'organization_project_count': 'โครงการทั้งหมด',
        'project_share_pct': 'สัดส่วนจำนวนโครงการ (%)',
        'value_share_pct': 'สัดส่วนวงเงิน (%)'
    })
)

priority_project_table = (
    priority_projects[
        [
            project_id_column,
            project_name_column,
            'pattern_name',
            'organization_name',
            'supplier_name',
            transaction_date_column,
            budget_column,
            'project_share_pct',
            'value_share_pct'
        ]
    ]
    .round({
        'project_share_pct': 2,
        'value_share_pct': 2
    })
    .rename(columns={
        project_id_column: 'รหัสโครงการ',
        project_name_column: 'ชื่อโครงการ',
        'pattern_name': 'Pattern',
        'organization_name': 'หน่วยงาน',
        'supplier_name': 'ผู้รับจ้าง',
        transaction_date_column: 'วันที่เกิดรายการ',
        budget_column: 'วงเงินงบประมาณ (บาท)',
        'project_share_pct': 'สัดส่วนจำนวนโครงการ (%)',
        'value_share_pct': 'สัดส่วนวงเงิน (%)'
    })
)

display(priority_group_table.head(20))
display(priority_project_table.head(20))

print(
    'โครงการตรวจสอบลำดับแรก: '
    f'{len(priority_project_ids):,}'
)


### อ่านผลลัพธ์และตรวจต่ออย่างไร

โครงการตรวจสอบลำดับแรกคือโครงการที่เข้า Pattern 1 และเข้า Pattern 2 หรือ Pattern 3 อย่างน้อยหนึ่งข้อ

ตารางแรกใช้เลือกคู่หน่วยงาน–ผู้รับจ้างที่จะตรวจ โดยเรียงตามสัดส่วนจำนวนโครงการ ตารางที่สองใช้เปิดดูรายโครงการ วันที่เกิดรายการแสดงเพื่อค้นเอกสารเท่านั้น


## 6. สรุป

- Pattern 1: โครงการใกล้เพดานที่เกิดซ้ำ
- Pattern 2: การพึ่งพาผู้รับจ้างในคอลัมน์ชื่อหน่วยงาน
- Pattern 3: การพึ่งพาผู้รับจ้างในคอลัมน์ชื่อหน่วยงานย่อย
- Pattern 4: การพึ่งพาผู้รับจ้างในคอลัมน์จังหวัด
- โครงการตรวจสอบลำดับแรกต้องเข้า Pattern 1 และเข้า Pattern 2 หรือ Pattern 3 อย่างน้อยหนึ่งข้อ


## 6. บันทึกผลลัพธ์

In [ ]:
project_flags = project_data[
    [project_id_column, 'in_study_scope']
].copy()

project_flags['flag_pattern_1'] = (
    project_flags[project_id_column].isin(pattern1_project_ids)
)
project_flags['flag_pattern_2'] = (
    project_flags[project_id_column].isin(pattern2_project_ids)
)
project_flags['flag_pattern_3'] = (
    project_flags[project_id_column].isin(pattern3_project_ids)
)
project_flags['flag_pattern_4'] = (
    project_flags[project_id_column].isin(pattern4_project_ids)
)

project_flags['priority_review'] = (
    project_flags[project_id_column].isin(priority_project_ids)
)

output_objects = {
    'project_review_indicators_2569.csv': project_flags,
    'priority_review_groups_2569.csv': priority_groups,
    'priority_review_projects_2569.csv': priority_projects,
    'repeated_near_500k_clusters_2569.csv': repeated_clusters,
    'agency_supplier_dependence_2569.csv': pattern2_relationships,
    'subagency_supplier_dependence_2569.csv': pattern3_relationships,
    'province_supplier_dependence_2569.csv': pattern4_relationships
}

export_records = []

for file_name, output_data in output_objects.items():
    output_path = processed_dir / file_name
    output_data.to_csv(
        output_path,
        index=False,
        encoding='utf-8-sig'
    )
    export_records.append({
        'file_name': file_name,
        'rows': len(output_data),
        'file_size_mb': output_path.stat().st_size / 1024**2
    })

export_summary = pd.DataFrame(export_records)
display(export_summary)
